# GemVision — Gemstone CNN Training (Colab GPU)

Trains the EfficientNetB0 gemstone-type classifier for the GemVision project.
Mirrors `ml/train_cnn.py` in the repo, just adapted to run here on a GPU.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> **T4 GPU**.

**Steps:** Run all cells top to bottom. Cell 2 will prompt you to upload
`gemstones-images.zip` (from `ml/data/gemstones-images.zip` in the project).
At the end, the two output files will download automatically to your browser's
Downloads folder — move them into `backend/models/` in the project.

In [ ]:
import tensorflow as tf
print('GPU available:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU found -- set Runtime > Change runtime type > T4 GPU, then re-run.'

In [ ]:
from google.colab import files
import zipfile, os

print('Select gemstones-images.zip from ml/data/ in the project...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('data')

DATA_DIR = 'data/gemstones-images'
print(os.listdir(DATA_DIR))

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
FROZEN_EPOCHS = 20
FINE_TUNE_EPOCHS = 15
FINE_TUNE_LR = 1e-5
FINE_TUNE_UNFREEZE_LAYERS = 30  # only the last ~block of EfficientNetB0, not the whole 237-layer backbone

train_dir = f'{DATA_DIR}/train'
test_dir = f'{DATA_DIR}/test'

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, label_mode='categorical'
)
class_names = train_ds.class_names
num_classes = len(class_names)
print(f'Found {num_classes} classes')

val_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, label_mode='categorical'
)

# No manual rescaling: EfficientNetB0 has its own preprocessing built in and
# expects raw [0, 255] float pixels. Adding a Rescaling(1/255) here as well
# double-shrinks the signal and the model never learns (this bit us once
# already in the local CPU run -- see README/commit history).
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds_eval = val_ds
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(
    include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), pooling='avg'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = layers.RandomFlip('horizontal')(inputs)
x = layers.RandomRotation(0.15)(x)
x = layers.RandomZoom(0.15)(x)
x = layers.RandomBrightness(0.15)(x)
# training=False is intentional and permanent here: keeps BatchNorm in
# inference mode through both the frozen phase and the fine-tuning phase
# below (standard Keras recipe for fine-tuning without wrecking pretrained
# BN statistics), independent of base_model.trainable at fit time.
x = base_model(x, training=False)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

print(f'=== Phase 1: frozen backbone, {FROZEN_EPOCHS} epochs ===')
history = model.fit(train_ds, validation_data=val_ds, epochs=FROZEN_EPOCHS, callbacks=callbacks)
combined_history = {k: list(v) for k, v in history.history.items()}

# Phase 1's EarlyStopping(restore_best_weights=True) already leaves `model`
# holding phase 1's own best-val_loss weights. Snapshot those (and the
# val_loss they achieved) so that if fine-tuning regresses -- a real risk
# with only ~2,856 images across 87 classes -- we can fall back to them
# instead of shipping a worse model than phase 1 alone.
phase1_weights = model.get_weights()
phase1_best_val_loss = min(history.history.get('val_loss', history.history['loss']))

In [ ]:
print(f'=== Phase 2: fine-tuning last {FINE_TUNE_UNFREEZE_LAYERS} layers, {FINE_TUNE_EPOCHS} epochs at lr={FINE_TUNE_LR} ===')
base_model.trainable = True
if FINE_TUNE_UNFREEZE_LAYERS > 0:
    for layer in base_model.layers[:-FINE_TUNE_UNFREEZE_LAYERS]:
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), loss='categorical_crossentropy', metrics=['accuracy'])

fine_tune_history = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks)
for k, v in fine_tune_history.history.items():
    combined_history.setdefault(k, []).extend(v)

phase2_best_val_loss = min(fine_tune_history.history.get('val_loss', fine_tune_history.history['loss']))
if phase2_best_val_loss > phase1_best_val_loss:
    print(f'Fine-tuning did not beat phase 1 (val_loss {phase2_best_val_loss:.4f} vs {phase1_best_val_loss:.4f}) -- keeping phase 1 weights instead.')
    model.set_weights(phase1_weights)
else:
    print(f'Fine-tuning improved on phase 1 (val_loss {phase2_best_val_loss:.4f} vs {phase1_best_val_loss:.4f}) -- keeping fine-tuned weights.')

In [ ]:
import json, numpy as np
from sklearn.metrics import classification_report

model.save('gemstone_cnn.keras')

class_indices = {name: idx for idx, name in enumerate(class_names)}
with open('class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)

y_true, y_pred = [], []
for batch_x, batch_y in val_ds_eval:
    preds = model.predict(batch_x, verbose=0)
    y_true.extend(np.argmax(batch_y.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print(report)
with open('cnn_classification_report.txt', 'w') as f:
    f.write(report)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(combined_history['accuracy'], label='train')
axes[0].plot(combined_history['val_accuracy'], label='val')
axes[0].axvline(FROZEN_EPOCHS - 1, color='gray', linestyle='--', label='fine-tune starts')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[1].plot(combined_history['loss'], label='train')
axes[1].plot(combined_history['val_loss'], label='val')
axes[1].set_title('Loss')
axes[1].legend()
fig.tight_layout()
fig.savefig('cnn_training_history.png')
plt.show()

## Download the results
The next cell downloads the two files the FastAPI backend actually needs.
Move both into `backend/models/` in the project (replacing any placeholder
files there). The report + plot are downloaded too, for your evaluation
chapter (proposal Objective 7).

In [ ]:
files.download('gemstone_cnn.keras')
files.download('class_indices.json')
files.download('cnn_classification_report.txt')
files.download('cnn_training_history.png')